# Experiment 01.01 — BTW Sandpile SOC Signature Validation

Validate diagnostic functions against the Bak-Tang-Wiesenfeld sandpile, the canonical SOC system. The goal is to confirm that our signature detection tools correctly identify SOC properties in a system where criticality is guaranteed by construction.

**Reference design:** [`work/experiments/validation/01_sandpile.md`](experiments/validation/01_sandpile.md)

**Implementation:** [`work/experiments/validation/sandpile.jl`](experiments/validation/sandpile.jl), [`work/experiments/validation/diagnostics.jl`](experiments/validation/diagnostics.jl)

## Setup

In [ ]:
include("experiments/validation/load_validation.jl")

using DataFrames
using Statistics
using Printf

## 1. Smoke test — small lattice

Run a tiny BTW sandpile to verify the simulator works end-to-end before doing anything expensive.

In [ ]:
smoke = btw_sandpile(32; N_transient=10_000, N_record=10_000, seed=1)

smoke_sizes = avalanche_sizes(smoke.catalog)

println("Smoke test — L=32, N_record=10,000:")
@printf("  Mean lattice height: %.3f (expected ~2.125 for BTW)\n", mean(smoke.final_height))
@printf("  Non-empty avalanches: %d / %d\n", length(smoke_sizes), length(smoke.catalog))
@printf("  Mean avalanche size:  %.2f\n", mean(smoke_sizes))
@printf("  Max avalanche size:   %d\n", maximum(smoke_sizes))

## 2. Production runs at L = 64, 128, 256

Per the experiment design, transient should be ~10 × L² grains. For L=256 that is ~650K grains. Recording phase is 10⁶ avalanches.

**Time estimate:** L=256 with 10⁶ avalanches may take many minutes. Start with L=64 and L=128 for initial validation; rerun at L=256 once tools are confirmed working.

In [ ]:
# Start small. Increase N_record once everything is verified.
L_values = [64, 128]
N_record = 200_000  # bump to 1_000_000 for production

results = Dict{Int, NamedTuple}()

for L in L_values
    N_transient = 10 * L * L
    @printf("L=%d: transient=%d, record=%d ... ", L, N_transient, N_record)
    t = @elapsed res = btw_sandpile(L; N_transient=N_transient,
                                       N_record=N_record, seed=42)
    results[L] = res
    @printf("done in %.1fs (mean height = %.3f)\n",
            t, mean(res.final_height))
end

### Catalog summary

In [ ]:
summary_rows = NamedTuple[]
for L in L_values
    sizes = avalanche_sizes(results[L].catalog)
    durations = avalanche_durations(results[L].catalog)
    push!(summary_rows, (
        L = L,
        n_recorded = length(results[L].catalog),
        n_nonempty = length(sizes),
        mean_size = mean(sizes),
        median_size = median(sizes),
        max_size = maximum(sizes),
        mean_duration = mean(durations),
        max_duration = maximum(durations),
        mean_height = mean(results[L].final_height),
    ))
end
DataFrame(summary_rows)

## 3. Signature 1 — Power-law avalanche size distribution

Expected: tau_s ≈ 1.2 (effective; BTW exhibits multiscaling per Tebaldi et al. 1999, so a single exponent is approximate).

In [ ]:
pl_rows = NamedTuple[]
for L in L_values
    sizes = avalanche_sizes(results[L].catalog)
    fit = fit_power_law(Float64.(sizes))
    cmp = compare_power_law_exponential(Float64.(sizes), fit.xmin, fit.alpha)
    push!(pl_rows, (
        L = L,
        xmin = fit.xmin,
        alpha = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
        ll_ratio_pl_vs_exp = cmp.log_likelihood_ratio,
    ))
end
DataFrame(pl_rows)

Same fit for duration distribution (tau_t expected ~1.4-1.5):

In [ ]:
dur_rows = NamedTuple[]
for L in L_values
    durs = avalanche_durations(results[L].catalog)
    fit = fit_power_law(Float64.(durs))
    push!(dur_rows, (
        L = L,
        xmin = fit.xmin,
        alpha_duration = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
    ))
end
DataFrame(dur_rows)

## 4. Signature 2 — Spatial correlation in the height field

G(r) should decay as a power law (with logarithmic corrections in BTW), not exponentially. Expensive at large L; sample if needed.

In [ ]:
# Only compute for L=64 (correlation_function is O(L^4))
corr64 = correlation_function(results[64].final_height; max_r=16)
println("Height-height correlation G(r) at L=64:")
for (r, g, n) in zip(corr64.r, corr64.G, corr64.n_pairs)
    @printf("  r=%2d  G(r)=% .4f  (n_pairs=%d)\n", r, g, n)
end

## 5. Signature 3 — Spectral analysis

Compute PSD of the avalanche size time series. **Note:** BTW does NOT produce clean 1/f noise. Expected high-frequency exponent ~1.56 with three distinct frequency regimes (Chhimpa et al. 2025).

In [ ]:
spec_rows = NamedTuple[]
for L in L_values
    # Use full sequence including empty avalanches (preserves time structure)
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    spec = power_spectrum(Float64.(series))
    beta_fit = spectral_exponent(spec.freq, spec.psd)
    h = hurst_rs(Float64.(series))
    push!(spec_rows, (
        L = L,
        beta = beta_fit.beta,
        beta_n_used = beta_fit.n_used,
        hurst = h.H,
        hurst_n_scales = length(h.scales),
    ))
end
DataFrame(spec_rows)

## 6. Signature 4 — Fractal structure of avalanche frontiers

BTW avalanche clusters are approximately compact (D ≈ 2). The fractal property is in the **frontier** (boundary), expected D_frontier ≈ 1.25 (Moghimi-Araghi et al. 2009).

For now we measure the cluster footprint dimension; frontier extraction can be added later.

In [ ]:
# Re-run a small set of avalanches with site tracking would require modifying
# the simulator. The current AvalancheRecord only stores aggregate stats.
# For now: report the size-area scaling exponent as a proxy.
#
# s ~ a^gamma where gamma > 1 indicates multiple topplings per site (BTW signature)

fractal_rows = NamedTuple[]
for L in L_values
    catalog = results[L].catalog
    nonempty = [a for a in catalog if a.size > 0]
    sizes = Float64.([a.size for a in nonempty])
    areas = Float64.([a.area for a in nonempty])
    # Fit log(size) vs log(area) on the upper half (large avalanches)
    big_mask = sizes .>= quantile(sizes, 0.5)
    log_s = log.(sizes[big_mask])
    log_a = log.(areas[big_mask])
    mean_x = mean(log_a)
    mean_y = mean(log_s)
    gamma = sum((log_a .- mean_x) .* (log_s .- mean_y)) / sum((log_a .- mean_x).^2)
    push!(fractal_rows, (
        L = L,
        size_area_exponent_gamma = gamma,
        n_used = sum(big_mask),
    ))
end
DataFrame(fractal_rows)

## 7. Signature 5 — Fat-tailed changes

Excess kurtosis of first differences in rolling avalanche activity. Expected: K > 1.5.

In [ ]:
kurt_rows = NamedTuple[]
for L in L_values
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    raw = fat_tail_kurtosis(Float64.(series); window=1)
    win100 = fat_tail_kurtosis(Float64.(series); window=100)
    win1000 = fat_tail_kurtosis(Float64.(series); window=1000)
    push!(kurt_rows, (
        L = L,
        excess_kurtosis_raw = raw.excess_kurtosis,
        excess_kurtosis_w100 = win100.excess_kurtosis,
        excess_kurtosis_w1000 = win1000.excess_kurtosis,
    ))
end
DataFrame(kurt_rows)

## 8. Branching ratio (activity-dependent)

Per Michiels van Kessenich et al. (2010): a single global b is misleading. Look for a **broad activity range where b(x) ≈ 1** — that is the SOC signature.

In [ ]:
for L in L_values
    bx = branching_ratio_activity_dependent(results[L].catalog; n_bins=20)
    global_b = branching_ratio_global(results[L].catalog)
    println("L=$L:")
    @printf("  Global mean b: %.3f\n", global_b)
    println("  Activity-dependent b(x):")
    for (a, b, n) in zip(bx.activity_levels, bx.b_of_x, bx.n_in_bin)
        @printf("    activity ~ %7.1f   b(x) = %.3f   (n=%d)\n", a, b, n)
    end
    println()
end

## 9. Inter-event time distribution

Waiting times between avalanches above selected thresholds. Expected: power-law or stretched exponential at high thresholds, no characteristic waiting time.

In [ ]:
for L in L_values
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    nonempty = avalanche_sizes(results[L].catalog)
    println("L=$L thresholds and waiting time stats:")
    for q in [0.50, 0.90, 0.99]
        thr = quantile(Float64.(nonempty), q)
        waits = inter_event_times(Float64.(series), thr)
        if length(waits) >= 10
            @printf("  q=%.2f  thr=%.0f  n_waits=%d  mean=%.1f  median=%.1f  max=%d\n",
                    q, thr, length(waits), mean(waits), median(waits), maximum(waits))
        else
            @printf("  q=%.2f  thr=%.0f  n_waits=%d  (insufficient)\n",
                    q, thr, length(waits))
        end
    end
    println()
end

## 10. Summary against published values

Compare what we measured to literature for 2D BTW. See [`01_sandpile.md`](experiments/validation/01_sandpile.md) §References for sources.

| Quantity | Expected | Status |
|----------|----------|--------|
| Mean height | 2.125 (exact, Dhar 1999) | Compare to `mean_height` above |
| tau_s (size) | ~1.2 effective; multiscaling | Compare to `alpha` in Section 3 |
| tau_t (duration) | ~1.4-1.5 | Compare to `alpha_duration` in Section 3 |
| PSD beta | ~1.56 high-freq (NOT 1/f) | Compare to `beta` in Section 5 |
| Hurst H | > 0.5 (persistent) | Compare to `hurst` in Section 5 |
| Excess kurtosis | > 1.5 | Compare in Section 7 |
| Branching b(x) | broad regime ≈ 1 | Compare in Section 8 |

**Next steps when this passes:**
- Increase to L=256 and N_record=10⁶ for production exponents
- Add negative controls (Poisson, subcritical, supercritical)
- Implement frontier extraction for true fractal dimension measurement
- Multiscaling test: moment ratios across L values
- Add Manna model for universality-class comparison (deferred per Experiment 01.02)